In [1]:
import ast
import numpy as np
import pandas as pd

In [2]:
#עמודות Data Leakage
LEAKAGE_COLUMNS = ["averageRating", "numVotes", "BoxOffice"]

In [3]:
#בונים פונקציה שסופרת כמות שחקנים בסרט
def count_actors(actors_raw):
    if pd.isna(actors_raw):
        return 0
    try:
        actors_list = ast.literal_eval(actors_raw)
        return len(actors_list)
    except (ValueError, SyntaxError):
        # במידה והערך בטבלה לא רשימה
        return 0

In [5]:
def clean_genres(genres_raw):
    # ניקיון של סוגריים: ['Drama'] -> Drama. 
    #גם טיפול בערכים שלא עומדים בתנאי של LIST
    if pd.isna(genres_raw):
        return ""
    cleaned = genres_raw.replace("[", "").replace("]", "")
    cleaned = cleaned.replace("'", "").replace('"', "")
    parts = [part.strip() for part in cleaned.split(",")]
    return ",".join(parts)


def prepare_data(df):
    movies_df = df.copy()

    # טיפול בג'אנרים
    movies_df["genres"] = movies_df["genres"].apply(clean_genres)

    # startYear = 0    זה טעות, 0 יגרום למודל לטעות.
    movies_df["startYear"] = movies_df["startYear"].replace(0, np.nan)
    
    #פיטצר חדש: חישוב - כמות גאנרים בכל סרט וסרט
    movies_df["num_genres"] = movies_df["genres"].apply(
        lambda g: len(g.split(",")) if g else 0
    )

    # פיטצר חדש: חישוב - גאנר ראשי
    movies_df["main_genre"] = movies_df["genres"].apply(
        lambda g: g.split(",")[0] if g else "Unknown"
    )

    # פיטצר חדש: חישוב - באקרטינג לפי משך הסרט
    runtime_bins   = [0, 70, 85, 100, 120, 150, 400]
    runtime_labels = [1, 2, 3, 4, 5, 6]
    
    movies_df["runtime_bin"] = pd.cut(
        movies_df["runtimeMinutes"], bins=runtime_bins, labels=runtime_labels
    ).astype("float")

    #פיטצר חדש: יחס משך הסרט / מספר גאנרים בסרט   
    movies_df["runtime_per_genre"] = np.where(
        movies_df["num_genres"] > 0,
        movies_df["runtimeMinutes"] / movies_df["num_genres"],
        np.nan
    )
    #פיטצר חדש: משתמשים בפונקציה שבנינו מקודם ומחשבים כמה שחקנים בכל סרט
    movies_df["num_actors"] = movies_df["lead_actors_ids"].apply(count_actors)
    #פיטצר חדש (בינארי): האם בכלל קיימים שחקנים בסרט
    movies_df["is_no_cast"] = (movies_df["num_actors"] == 0).astype(int)


    #פיטצר חדש: כמה מילים יש בכל סרט
    movies_df["title_word_count"] = (
        movies_df["primaryTitle"].fillna("").apply(lambda t: len(t.split()))
    )

    #פיטצר חדש: האם יש נקודתיים בשם הסרט
    movies_df["title_has_colon"] = (
        movies_df["primaryTitle"].fillna("").str.contains(":").astype(int)
    )
    #פיטצר חדש: האם שפה של סרט הוא אנגלית
    movies_df["is_english"] = (movies_df["Language"] == "English").astype(int)
    #פיטצר חדש - האם סרט נוצר בארצות הברית
    movies_df["is_us"]      = (movies_df["Country"] == "United States").astype(int)

    #פיטצר חדש: האם קיים תקציב עבור כל סרט
    movies_df["has_budget"] = movies_df["budget"].notna().astype(int)

    feature_columns = [
        "startYear", "runtimeMinutes",
        "num_genres", "main_genre",
        "runtime_bin", "runtime_per_genre",
        "num_actors", "is_no_cast",
        "title_word_count", "title_has_colon",
        "is_english", "is_us",
        "has_budget",
    ]
    return movies_df[feature_columns]

<div dir="rtl"> 
בדיקה: האם פונקציה שבנינו עובדת

In [7]:
df = pd.read_csv("dataset.csv", low_memory=False)
data_prepared = prepare_data(df)

In [9]:
data_prepared.sample(5)

,startYear,runtimeMinutes,num_genres,main_genre,runtime_bin,runtime_per_genre,num_actors,is_no_cast,title_word_count,title_has_colon,is_english,is_us,has_budget
10135,2018.0,100.0,3,Comedy,3.0,33.333333,5,0,8,0,0,0,0
111121,2013.0,96.0,1,Documentary,3.0,96.000000,0,1,7,0,0,0,0
97345,1967.0,90.0,2,Comedy,3.0,45.000000,5,0,4,0,0,0,0
65534,2023.0,127.0,1,Drama,5.0,127.000000,3,0,3,0,0,0,0
33467,2019.0,60.0,1,Documentary,1.0,60.000000,1,0,3,0,0,0,0
